<a href="https://colab.research.google.com/github/Aniyarath/AI-APPS/blob/main/day1_demo_code_vectorDB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FinInsight AI – Pipeline with Vector Database (FAISS)
End-to-end: Upload → Extract → Clean → Excel → Chunk → Embed → Vector DB → Retrieve

In [1]:
from google.colab import files
uploaded = files.upload()

Saving bank_statement_5.pdf to bank_statement_5.pdf
Saving bank_statement_4.pdf to bank_statement_4.pdf
Saving bank_statement_3.pdf to bank_statement_3.pdf
Saving bank_statement_2.pdf to bank_statement_2.pdf
Saving bank_statement_1.pdf to bank_statement_1.pdf
Saving receipt_image_3.png to receipt_image_3.png
Saving receipt_image_2.png to receipt_image_2.png
Saving receipt_image_1.png to receipt_image_1.png


In [2]:
!pip install pdfplumber pytesseract sentence-transformers pandas openpyxl faiss-cpu
!apt-get install -y tesseract-ocr

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 72.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 72.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 30.1 MB/s eta 0:00:00
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.


## Extract from PDFs

In [3]:
import pdfplumber
pdf_texts=[]
for filename in uploaded:
    if filename.endswith('.pdf'):
        with pdfplumber.open(filename) as pdf:
            text=''
            for p in pdf.pages:
                text+=p.extract_text() or ''
        pdf_texts.append(text)

## Extract from Images (OCR)

In [4]:
import pytesseract
from PIL import Image
image_texts=[]
for filename in uploaded:
    if filename.endswith('.png') or filename.endswith('.jpg'):
        img=Image.open(filename)
        image_texts.append(pytesseract.image_to_string(img))

## Clean Data

In [6]:
import re, pandas as pd
lines=[]
for t in pdf_texts:
    lines+=t.split('\n')
clean=[]
for l in lines:
    #match=re.findall(r'(Apr \d+)\s+(.+?)\s+(.+?)\s+\$(\d+\.\d+)',l)
    match = re.findall(r'(Apr \d+)\s+(\S+(?:\s+\S+)?)\s+(.+?)\s+\$(\d+\.\d+)', l)
    for m in match:
        clean.append(m)
df=pd.DataFrame(clean,columns=['Date','Merchant','Category','Amount'])
df['Amount']=df['Amount'].astype(float)
df.head()

,Date,Merchant,Category,Amount
0,Apr 21,Best Buy,Shopping,142.29
1,Apr 13,Apple Store,Shopping,100.31
2,Apr 20,Apple Store,Shopping,101.50
3,Apr 20,Uber,Transport,291.30
4,Apr 14,Uber,Transport,129.09


## Save Excel

In [7]:
df.to_excel('cleaned_data.xlsx',index=False)

## Chunking

In [8]:
chunks=df.apply(lambda r: f'I spent ${r['Amount']} on {r['Category']} at {r['Merchant']} on {r['Date']}', axis=1).tolist()
chunks[:5]

['I spent $142.29 on Shopping at Best Buy on Apr 21',
 'I spent $100.31 on Shopping at Apple Store on Apr 13',
 'I spent $101.5 on Shopping at Apple Store on Apr 20',
 'I spent $291.3 on Transport at Uber on Apr 20',
 'I spent $129.09 on Transport at Uber on Apr 14']

## Embeddings

In [9]:
from sentence_transformers import SentenceTransformer
model=SentenceTransformer('all-MiniLM-L6-v2')
embeddings=model.encode(chunks)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## Vector Database (FAISS)

In [10]:
import faiss, numpy as np
vecs=np.array(embeddings).astype('float32')
dim=vecs.shape[1]
index=faiss.IndexFlatL2(dim)
index.add(vecs)

## Retrieval

In [12]:
query='food spending'
q_vec=model.encode([query]).astype('float32')
k=5
dist, idx=index.search(q_vec,k)
for i in idx[0]:
    print(chunks[i])

I spent $267.45 on Food at Chipotle on Apr 10
I spent $149.35 on Food at Chipotle on Apr 19
I spent $267.51 on Food at Starbucks on Apr 22
I spent $188.82 on Food at Chipotle on Apr 25
I spent $296.88 on Food at Chipotle on Apr 25
